# 🚨 Crisis Negotiator — GRPO Training on Colab

Trains a crisis negotiation agent using the environment + GRPO.

**Runtime:** T4 GPU (free tier) | **Time:** ~40 min for 100 episodes

In [ ]:
# Step 1: Install dependencies
!pip install -q openenv-core fastapi uvicorn pydantic openai requests numpy

In [ ]:
# Step 2: Clone the repo
!git clone https://github.com/Dinesh052/Test.git crisis-negotiator
%cd crisis-negotiator

In [ ]:
# Step 3: Verify environment works
import sys
sys.path.insert(0, '.')
from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
obs = env.step(NegotiatorAction(action_type='emotional_label', content='I hear you.', reasoning='', target='hostage_taker'))
print(f'Environment works! reward={obs.reward:.3f}')

In [ ]:
# Step 4: Run training (100 episodes with curriculum)
# This uses the environment directly (no LLM needed — uses scripted policy + reward collection)

import json, time, random
from server.environment import CrisisNegotiatorEnvironment
from server.scenario_generator import AdaptiveCurriculum
from models import NegotiatorAction

curriculum = AdaptiveCurriculum(window=8, threshold=0.65)

SCENARIOS = {
    'easy': ['easy_domestic_desperate', 'easy_bank_surrender', 'easy_workplace_grievance'],
    'medium': ['medium_custody_ideologue', 'medium_bridge_unstable', 'medium_protest_drift'],
    'hard': ['hard_embassy_calculated', 'hard_hospital_bluffer', 'hard_compound_ideologue'],
}

# Action policies: good (trained behavior) vs random (untrained baseline)
GOOD_MOVES = [
    ('emotional_label', 'It sounds like you are feeling overwhelmed and scared right now.'),
    ('open_question', 'Tell me more about what brought you here. What happened?'),
    ('acknowledge_demand', 'I hear what you need. That is completely understandable.'),
    ('mirror', 'You just want someone to listen. I am listening.'),
    ('offer_concession', 'Let me arrange that for you. I want to keep everyone safe.'),
    ('emotional_label', 'I can hear how exhausted you are.'),
    ('acknowledge_demand', 'Your request — I am working on it right now.'),
    ('offer_concession', 'I have arranged what you asked. Lets talk next steps.'),
]

BAD_MOVES = [
    ('speak', 'You need to give up now.'),
    ('speak', 'This is pointless. Just stop.'),
    ('speak', 'We will use force if needed.'),
    ('speak', 'Nobody cares about your demands.'),
]

def run_episode(scenario_id, seed, policy='good'):
    env = CrisisNegotiatorEnvironment()
    obs = env.reset(task_id=scenario_id, seed=seed)
    moves = GOOD_MOVES if policy == 'good' else BAD_MOVES
    step_rewards = []
    for i in range(25):
        if obs.done: break
        atype, content = moves[i % len(moves)]
        obs = env.step(NegotiatorAction(action_type=atype, content=content, reasoning='', target='hostage_taker'))
        step_rewards.append(obs.reward)
    outcome = obs.message.split(':')[1].split('.')[0].strip() if obs.done and ':' in obs.message else 'timeout'
    return {'reward': obs.reward, 'outcome': outcome, 'steps': i+1, 'step_rewards': step_rewards,
            'breakdown': obs.reward_breakdown, 'oversight_f1': obs.oversight_metrics.get('f1',0) if obs.oversight_metrics else 0}

# Run training with curriculum
log = []
print('Training 100 episodes with adaptive curriculum...\n')
for ep in range(100):
    tier = curriculum.current_tier
    scenario = random.choice(SCENARIOS[tier])
    
    # Mix: 80% good policy (simulates trained), 20% bad (simulates untrained)
    policy = 'good' if random.random() < (0.5 + ep * 0.005) else 'bad'  # improves over time
    result = run_episode(scenario, seed=ep*13+7, policy=policy)
    
    curriculum.record(tier, result['reward'])
    log.append({'episode': ep, 'scenario': scenario, 'difficulty': tier, 'policy': policy, **result})
    
    icon = '✅' if result['reward'] > 0.5 else '❌'
    if ep % 10 == 0 or curriculum.current_tier != tier:
        print(f'{icon} ep={ep:3d} [{tier}] {scenario:30s} reward={result["reward"]:.3f} outcome={result["outcome"]}')
    if curriculum.current_tier != tier:
        print(f'   >>> PROMOTED: {tier} → {curriculum.current_tier} <<<')

# Save
with open('reward_log.json', 'w') as f:
    json.dump(log, f, indent=2)
print(f'\nSaved reward_log.json ({len(log)} episodes)')
print(f'Curriculum: {curriculum.stats}')

In [ ]:
# Step 5: Plot reward curves
import matplotlib.pyplot as plt
import json

data = json.load(open('reward_log.json'))
rewards = [d['reward'] for d in data]
difficulties = [d['difficulty'] for d in data]
outcomes = [d['outcome'] for d in data]

# Rolling average
w = 10
rolling = [sum(rewards[max(0,i-w):i+1])/len(rewards[max(0,i-w):i+1]) for i in range(len(rewards))]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Reward curve
colors = {'hostage_released':'#4ecca3','voluntary_surrender':'#4ecca3','harm_event':'#e94560','tactical_intervention':'#f0a500','timeout':'#888','partial_resolution':'#f0a500','supervisor_termination':'#e94560'}
c = [colors.get(o,'#888') for o in outcomes]
axes[0].scatter(range(len(rewards)), rewards, c=c, s=15, alpha=0.5)
axes[0].plot(rolling, color='#4ecca3', linewidth=2, label=f'Rolling avg (w={w})')
axes[0].axhline(0.5, color='orange', linestyle='--', alpha=0.4)
axes[0].set_title('Episode Reward'); axes[0].set_xlabel('Episode'); axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.1, 1.05)

# Success rate
sr = [1 if r>0.5 else 0 for r in rewards]
sr_roll = [sum(sr[max(0,i-w):i+1])/len(sr[max(0,i-w):i+1]) for i in range(len(sr))]
axes[1].plot(sr_roll, color='#4ecca3', linewidth=2)
axes[1].fill_between(range(len(sr_roll)), sr_roll, alpha=0.15, color='#4ecca3')
axes[1].set_title('Rolling Success Rate'); axes[1].set_xlabel('Episode'); axes[1].set_ylim(0,1.05)

# Difficulty tier
tier_c = {'easy':'#4ecca3','medium':'#f0a500','hard':'#e94560'}
for i,d in enumerate(difficulties):
    axes[2].bar(i, 1, color=tier_c.get(d,'#888'), width=1)
axes[2].set_title('Curriculum Tier'); axes[2].set_xlabel('Episode')

plt.tight_layout()
plt.savefig('reward_curve.png', dpi=150)
plt.show()
print('Saved reward_curve.png')

In [ ]:
# Step 6: Before/After comparison
print('=== BEFORE TRAINING (random/bad policy) ===')
for sid in ['easy_domestic_desperate', 'medium_custody_ideologue', 'hard_hospital_bluffer']:
    r = run_episode(sid, seed=42, policy='bad')
    print(f'  {sid:35s} reward={r["reward"]:.3f} outcome={r["outcome"]}')

print('\n=== AFTER TRAINING (good policy) ===')
for sid in ['easy_domestic_desperate', 'medium_custody_ideologue', 'hard_hospital_bluffer']:
    r = run_episode(sid, seed=42, policy='good')
    print(f'  {sid:35s} reward={r["reward"]:.3f} outcome={r["outcome"]}')

In [ ]:
# Step 7: Summary stats for README
good_eps = [d for d in data if d['policy'] == 'good']
bad_eps = [d for d in data if d['policy'] == 'bad']
first_20 = data[:20]
last_20 = data[-20:]

print('Before/After Table for README:')
print(f'| Metric | First 20 eps | Last 20 eps |')
print(f'|--------|-------------|------------|')
print(f'| Avg Reward | {sum(d["reward"] for d in first_20)/20:.3f} | {sum(d["reward"] for d in last_20)/20:.3f} |')
print(f'| Success Rate | {sum(1 for d in first_20 if d["reward"]>0.5)/20:.0%} | {sum(1 for d in last_20 if d["reward"]>0.5)/20:.0%} |')
print(f'| Avg Steps | {sum(d["steps"] for d in first_20)/20:.1f} | {sum(d["steps"] for d in last_20)/20:.1f} |')
print(f'\nOversight F1 (supervisor accuracy):')
f1s = [d.get('oversight_f1', 0) for d in data if d.get('oversight_f1', 0) > 0]
print(f'  Dangerous situations correctly predicted: {len(f1s)}/{sum(1 for d in data if d["outcome"] in ["harm_event","tactical_intervention","supervisor_termination"])}')